# GEDI STEP05 catalogue audit

This read-only notebook verifies the frozen post-QC catalogues used by training. It does **not** claim to regenerate the catalogues from raw GEDI granules.


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
ROOT = Path.cwd().resolve()
CATALOG_ROOT = ROOT / 'data' / 'processed' / 'gedi'
CATALOGS = {site: CATALOG_ROOT/site/'shot_catalog_step05.csv.gz' for site in ('ifran','maamoura','agadir')}
CATALOGS


In [ ]:
def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''):
            h.update(block)
    return h.hexdigest()

rows = []
for site, path in CATALOGS.items():
    frame = pd.read_csv(path, low_memory=False)
    required = {'split', 'aux_shot_uid', 'rh95'}
    assert required.issubset(frame.columns), (site, required - set(frame.columns))
    assert frame['split'].isin(['train','val','test']).all()
    unique = frame.drop_duplicates(['split','aux_shot_uid'])
    counts = unique['split'].value_counts()
    rows.append({'site': site, 'occurrence_rows': len(frame), 'unique_shots': len(unique), 'train': int(counts.get('train',0)), 'val': int(counts.get('val',0)), 'test': int(counts.get('test',0)), 'rh95_min_m': unique.rh95.min(), 'rh95_max_m': unique.rh95.max(), 'sha256': digest(path)})
audit = pd.DataFrame(rows)
display(audit)


The occurrence rows can exceed the unique-shot count because a GEDI shot may be paired with multiple eligible patch–time samples. Metrics must use their explicitly frozen evaluation-support tables rather than the full occurrence catalogue.
